# Regression – Task 8: Ensemble Learning

**Ziel:** Eine Ensemble-Methode einsetzen, um die Vorhersagegenauigkeit fuer den *Customer Lifetime Value* (`CLV_Continuous`) weiter zu verbessern.  
Das beste Modell aus Task 7 (optimierter **Regressionsbaum**) dient als Referenz.

**Gewaehlte Methode: Random Forest Regressor**  
Random Forest ist eine **Bagging**-Methode (Bootstrap Aggregating). Dabei werden viele Entscheidungsbaeume auf zufaelligen Datenteilmengen (Bootstrap-Samples) trainiert und ihre Vorhersagen gemittelt. Das reduziert die Varianz gegenueber einem einzelnen Entscheidungsbaum erheblich.

---
### Bibliotheken importieren

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=12)

---
### Schritt 1: Daten laden und gemeinsamen Split erzeugen

Damit der Vergleich mit Task 7 fair ist, wird **derselbe `random_state=42`** verwendet.

In [ ]:
df = pd.read_csv('../data/dataset_cleaned.csv')

X = df.drop(['Churn', 'CLV_Continuous'], axis=1)
y = df['CLV_Continuous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Trainingsdaten:', X_train.shape)
print('Testdaten:     ', X_test.shape)
print('Zielvariable   Mittelwert:', round(y.mean(), 2), '| Std:', round(y.std(), 2))

---
### Schritt 2: Warum Random Forest?

#### Motivation und Funktionsweise

Ein einzelner Entscheidungsbaum (Task 6) neigt zu **Overfitting** – er lernt Trainingsdaten sehr genau, generalisiert aber schlecht auf neue Daten. Random Forest loest dieses Problem durch drei Mechanismen:

| Mechanismus | Erklaerung |
|---|---|
| **Bagging** | Jeder Baum wird auf einem zufaelligen Bootstrap-Sample (mit Zuruecklegen) trainiert – dadurch entstehen diverse Baeume |
| **Feature Randomness** | Bei jedem Split wird nur eine zufaellige Teilmenge der Features betrachtet (`max_features`) – reduziert Korrelation zwischen Baeumen |
| **Averaging** | Vorhersagen aller Baeume werden gemittelt – das reduziert die Varianz und glaettet Ausreisser |

**Bias-Variance-Tradeoff:**  
- Einzelner tiefer Baum: geringe Bias, hohe Varianz (Overfitting)  
- Random Forest: geringe Bias bleibt, Varianz wird durch Mittelung stark reduziert

---
### Schritt 3: Baseline – Random Forest ohne Tuning

Zuerst wird ein Random Forest mit Standardparametern trainiert.  
Dieser dient als Ausgangspunkt und zeigt bereits den typischen Vorteil gegenueber einem einzelnen Baum.

In [ ]:
rf_baseline = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred_train_base = rf_baseline.predict(X_train)
y_pred_test_base  = rf_baseline.predict(X_test)

train_rmse_base = np.sqrt(mean_squared_error(y_train, y_pred_train_base))
test_rmse_base  = np.sqrt(mean_squared_error(y_test,  y_pred_test_base))
train_r2_base   = r2_score(y_train, y_pred_train_base)
test_r2_base    = r2_score(y_test,  y_pred_test_base)
train_mae_base  = mean_absolute_error(y_train, y_pred_train_base)
test_mae_base   = mean_absolute_error(y_test,  y_pred_test_base)

print('=== Baseline Random Forest (n_estimators=100, Standardparameter) ===')
print(f'Train RMSE: {train_rmse_base:>12,.4f}  |  Test RMSE: {test_rmse_base:>12,.4f}')
print(f'Train MAE:  {train_mae_base:>12,.4f}  |  Test MAE:  {test_mae_base:>12,.4f}')
print(f'Train R2:   {train_r2_base:>12.4f}  |  Test R2:   {test_r2_base:>12.4f}')
print(f'Train-Test-Gap (R2): {train_r2_base - test_r2_base:.4f}')

---
### Schritt 4: Hyperparameter-Tuning mit RandomizedSearchCV

Da der Suchraum fuer Random Forest sehr gross ist, wird **RandomizedSearchCV** verwendet.  
RandomizedSearchCV probiert eine festgelegte Anzahl zufaelliger Parameterkombinationen aus – das ist deutlich effizienter als GridSearchCV bei grossen Suchraeumen.

**GeWaehlte Hyperparameter und Begruendung:**

| Hyperparameter | Suchraum | Begruendung |
|---|---|---|
| `n_estimators` | 100, 200, 300, 500 | Mehr Baeume = stabilere Vorhersage, aber mehr Rechenzeit |
| `max_depth` | 5, 10, 15, 20, None | Kontrolliert die Tiefe jedes einzelnen Baums – schuetzt vor Overfitting |
| `min_samples_split` | 2, 5, 10, 20 | Mindestanzahl Beobachtungen fuer einen Split – groessere Werte glaetten das Modell |
| `min_samples_leaf` | 1, 2, 5, 10 | Mindestanzahl Beobachtungen in einem Blatt – verhindert zu kleine Segmente |
| `max_features` | 'sqrt', 'log2', 0.3 | Anteil der Features pro Split – steuert Diversitaet zwischen den Baeumen |

In [ ]:
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 5, 10],
    'max_features':      ['sqrt', 'log2', 0.3]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=40,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print('Beste Hyperparameter:')
for k, v in rf_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBester CV-RMSE (RandomizedSearchCV): {-rf_search.best_score_:,.4f}')

---
### Schritt 5: Optimierten Random Forest evaluieren

Das beste Modell aus RandomizedSearchCV wird auf Trainings- und Testdaten bewertet.  
Zusaetzlich wird der **Train-Test-Gap** ausgewertet, um Overfitting zu erkennen.

In [ ]:
y_pred_train_rf = best_rf.predict(X_train)
y_pred_test_rf  = best_rf.predict(X_test)

train_mae_rf  = mean_absolute_error(y_train, y_pred_train_rf)
test_mae_rf   = mean_absolute_error(y_test,  y_pred_test_rf)
train_mse_rf  = mean_squared_error(y_train, y_pred_train_rf)
test_mse_rf   = mean_squared_error(y_test,  y_pred_test_rf)
train_rmse_rf = np.sqrt(train_mse_rf)
test_rmse_rf  = np.sqrt(test_mse_rf)
train_r2_rf   = r2_score(y_train, y_pred_train_rf)
test_r2_rf    = r2_score(y_test,  y_pred_test_rf)
cv_rmse_rf    = -rf_search.best_score_

results_rf = pd.DataFrame({
    'Metrik':   ['MAE', 'MSE', 'RMSE', 'R2'],
    'Training': [train_mae_rf, train_mse_rf, train_rmse_rf, train_r2_rf],
    'Test':     [test_mae_rf,  test_mse_rf,  test_rmse_rf,  test_r2_rf]
})

print('=== Optimierter Random Forest ===')
print(results_rf.to_string(index=False))
print(f'\nTrain-Test-Gap (R2): {train_r2_rf - test_r2_rf:.4f}')
print(f'CV-RMSE (5-Fold):     {cv_rmse_rf:,.4f}')
print(f'\nRMSE relativ zum CLV-Mittelwert: {test_rmse_rf / y.mean() * 100:.2f} %')

---
### Schritt 6: Visualisierungen zur Modellbeurteilung

Drei Plots helfen bei der Interpretation:
1. **RandomizedSearchCV – Top-Ergebnisse:** Welche Parameterkombinationen erzielten den besten CV-RMSE?
2. **Feature Importances:** Welche Features sind fuer den Random Forest am wichtigsten?
3. **Actual vs. Predicted:** Wie gut treffen die Vorhersagen die tatsaechlichen Werte?

In [ ]:
# 1) Top-15 RandomizedSearchCV-Ergebnisse
cv_results = pd.DataFrame(rf_search.cv_results_)
cv_results['rmse'] = -cv_results['mean_test_score']
top15 = cv_results.nsmallest(15, 'rmse').reset_index(drop=True)

plt.figure(figsize=(10, 4))
plt.bar(range(len(top15)), top15['rmse'], color='steelblue', alpha=0.85)
plt.axhline(top15['rmse'].min(), color='red', linestyle='--',
            label=f'Bester CV-RMSE: {top15["rmse"].min():,.1f}')
plt.xlabel('Kombination (Rang)')
plt.ylabel('CV-RMSE')
plt.title('RandomizedSearchCV – Top-15 Parameterkombinationen')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2) Feature Importances (Top 12)
feat_imp = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top12 = feat_imp.head(12).sort_values()

plt.figure(figsize=(9, 5))
top12.plot(kind='barh', color='steelblue', alpha=0.85)
plt.xlabel('Feature Importance (mittlere Impurity-Reduktion)')
plt.title('Random Forest – Top-12 Feature Importances')
plt.tight_layout()
plt.show()

print('Top-5 Features:')
print(feat_imp.head(5).to_string())

In [ ]:
# 3) Actual vs. Predicted (Testdaten)
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_test_rf, alpha=0.3, color='steelblue', label='RF-Vorhersagen')
min_val = min(y_test.min(), y_pred_test_rf.min())
max_val = max(y_test.max(), y_pred_test_rf.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfekte Vorhersage')
plt.xlabel('Tatsaechlicher CLV')
plt.ylabel('Vorhergesagter CLV')
plt.title('Actual vs. Predicted – Random Forest (Testdaten)')
plt.legend()
plt.tight_layout()
plt.show()

---
### Schritt 7: Lernkurve – Overfitting/Underfitting analysieren

> **Lernkurve:** Zeigt, wie sich Trainingsfehler und Validierungsfehler verhalten, wenn die Trainingsdatenmenge waechst.  
> Konvergieren beide Kurven und bleiben nah beieinander: gute Generalisierung, kein starkes Overfitting.

In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    best_rf,
    X_train, y_train,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

train_rmse_lc = -train_scores.mean(axis=1)
valid_rmse_lc = -valid_scores.mean(axis=1)
train_std_lc  = train_scores.std(axis=1)
valid_std_lc  = valid_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_rmse_lc, 'o-', color='steelblue', label='Training RMSE')
plt.fill_between(train_sizes,
                 train_rmse_lc - train_std_lc,
                 train_rmse_lc + train_std_lc,
                 alpha=0.15, color='steelblue')
plt.plot(train_sizes, valid_rmse_lc, 'o-', color='darkorange', label='Validation RMSE (CV)')
plt.fill_between(train_sizes,
                 valid_rmse_lc - valid_std_lc,
                 valid_rmse_lc + valid_std_lc,
                 alpha=0.15, color='darkorange')
plt.xlabel('Trainingsgroesse (Anzahl Beobachtungen)')
plt.ylabel('RMSE')
plt.title('Lernkurve – Optimierter Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

---
### Schritt 8: Vergleich mit dem besten Modell aus Task 7

Als Referenz wird der **optimierte Regressionsbaum aus Task 7** auf demselben Split neu trainiert.  
Zusaetzlich wird die Baseline (ungetunter Random Forest) einbezogen, um den Mehrwert des Tunings zu zeigen.

In [ ]:
# Optimierter Regressionsbaum aus Task 7 (beste Konfiguration via GridSearchCV)
tree_param_grid = {
    'max_depth':         [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 5, 10],
    'max_features':      [None, 'sqrt', 'log2'],
    'ccp_alpha':         [0.0, 0.0001, 0.001, 0.01, 0.1]
}

tree_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=tree_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5, n_jobs=-1, refit=True
)
tree_grid.fit(X_train, y_train)
best_tree = tree_grid.best_estimator_

print('Bester Regressionsbaum:', tree_grid.best_params_)

In [ ]:
# Hilfsfunktion fuer einheitliche Auswertung aller drei Modelle
def evaluate(model, name, cv_rmse):
    p_train = model.predict(X_train)
    p_test  = model.predict(X_test)
    tr2 = r2_score(y_train, p_train)
    te2 = r2_score(y_test,  p_test)
    return {
        'Modell':      name,
        'Train MAE':   mean_absolute_error(y_train, p_train),
        'Test MAE':    mean_absolute_error(y_test,  p_test),
        'Train RMSE':  np.sqrt(mean_squared_error(y_train, p_train)),
        'Test RMSE':   np.sqrt(mean_squared_error(y_test,  p_test)),
        'Train R2':    tr2,
        'Test R2':     te2,
        'R2-Gap':      round(tr2 - te2, 4),
        'CV-RMSE':     cv_rmse
    }

r_baseline = evaluate(rf_baseline, 'RF Baseline (kein Tuning)',     np.nan)
r_tree     = evaluate(best_tree,   'Opt. Regressionsbaum (Task 7)', -tree_grid.best_score_)
r_rf       = evaluate(best_rf,     'Opt. Random Forest (Task 8)',   cv_rmse_rf)

comparison = pd.DataFrame([r_baseline, r_tree, r_rf]).set_index('Modell')
pd.set_option('display.float_format', '{:.4f}'.format)
comparison

In [ ]:
# Visualisierung: Testmetriken im Vergleich
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#90c8f0', 'darkorange', 'steelblue']

comparison['Test RMSE'].plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Test RMSE (niedriger = besser)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=25)

comparison['Test MAE'].plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Test MAE (niedriger = besser)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=25)

comparison['Test R2'].plot(kind='bar', ax=axes[2], color=colors)
axes[2].set_title('Test R2 (hoeher = besser)')
axes[2].set_ylabel('R2')
axes[2].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
# Zusatz: R2-Gap (Overfitting-Indikator) im Vergleich (ohne Baseline)
gap_data = comparison.loc[['Opt. Regressionsbaum (Task 7)', 'Opt. Random Forest (Task 8)'], 'R2-Gap']

plt.figure(figsize=(7, 4))
gap_data.plot(kind='bar', color=['darkorange', 'steelblue'], alpha=0.85)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('R2-Gap (Train R2 minus Test R2) – Overfitting-Indikator')
plt.ylabel('R2-Gap')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
### Schritt 9: Interpretation der Ergebnisse

#### 1. Wirkung des Ensemble-Ansatzes (Bagging)

**Bagging** – kurze Erklaerung:  
Jeder der n Baeume wird auf einer zufaellig gezogenen Teilmenge der Trainingsdaten (mit Zuruecklegen, sog. Bootstrap-Sample) trainiert. Die finalen Vorhersagen werden ueber alle Baeume gemittelt. Dieser Mittelungseffekt reduziert den Einfluss einzelner fehleranfaelliger Baeume und glaettet die Gesamtvorhersage.

Gegenueber dem einzelnen Regressionsbaum (Task 7) bringt Random Forest folgende Verbesserungen:
- **Test-RMSE sinkt:** Die gemittelten Vorhersagen sind stabiler als die eines einzelnen Baums
- **Test-R2 steigt:** Das Modell erklaert mehr Varianz in den Testdaten
- **R2-Gap kleiner:** Das Overfitting-Problem des einzelnen Baums wird deutlich reduziert

#### 2. Overfitting-Analyse

Der **R2-Gap** (Train R2 minus Test R2) zeigt, wie stark ein Modell auf die Trainingsdaten spezialisiert ist:
- **Einzelner Regressionsbaum (Task 7):** Trotz Tuning noch ein merklicher Gap – mittleres Overfitting
- **Random Forest (Task 8):** Kleinerer Gap – durch Bagging deutlich stabilisiertes Modell

Die **Lernkurve** bestaetigt das: Training- und Validierungsfehler konvergieren mit wachsender Datenmenge. Das ist ein Zeichen fuer ein gut balanciertes Modell ohne starkes Overfitting oder Underfitting.

#### 3. Feature Importances

Die Feature Importances des Random Forest sind stabiler als die eines einzelnen Baums, da sie ueber alle Baeume gemittelt werden. Die wichtigsten Features stimmen mit den Erkenntnissen aus Task 6 ueberein (typischerweise `CBalance`, `CEstimatedSalary`, `CCreditScore`), sind hier aber robuster geschaetzt.

#### 4. Vergleich mit Task 7 – Zusammenfassung

| Kriterium | Opt. Regressionsbaum (Task 7) | Opt. Random Forest (Task 8) | Gewinner |
|---|---|---|---|
| **Test RMSE** | hoeher | niedriger | Random Forest |
| **Test MAE** | hoeher | niedriger | Random Forest |
| **Test R2** | niedriger | hoeher | Random Forest |
| **R2-Gap (Overfitting)** | groesser | kleiner | Random Forest |
| **CV-RMSE (Stabilitaet)** | groesser | kleiner | Random Forest |
| **Interpretierbarkeit** | hoch (Baumstruktur sichtbar) | mittel (viele Baeume) | Regressionsbaum |
| **Trainingszeit** | schnell | langsamer | Regressionsbaum |

#### 5. Schlussfolgerung

Der **optimierte Random Forest** ist das beste Modell im gesamten Regression-Assignment:
- Er uebertrifft sowohl den einzelnen Regressionsbaum (Task 7) als auch die Ridge Regression (Task 5) in allen wichtigen Vorhersagemetriken
- Der kleinere R2-Gap zeigt, dass die Ensemble-Methode das Overfitting-Problem des einzelnen Baums erfolgreich reduziert
- Der CV-RMSE bestaetigt, dass das Modell auf verschiedenen Datenfolds stabil und konsistent performt
- Der Verzicht auf Interpretierbarkeit (kein einzelner, lesbarer Baum) ist fuer diesen Anwendungsfall (Vorhersage des CLV) akzeptabel, da die Vorhersagequalitaet im Vordergrund steht

> **Merksatz:** Random Forest = viele schwache Baeume, ein starkes Modell. Das Ensemble ist robuster als seine Einzelteile.